# Data Leakage Prevention (DLP) LoRA Fine-Tuning
**This notebook fine-tunes a lightweight model (Google Gemma-2b-it) using 4-bit quantization and LoRA to classify DLP actions.**

## Section 1 — Setup

Install dependencies and import libraries. Note: `google/gemma-2b-it` requires huggingface authentication. You might need to add a Kaggle Secret called `HF_TOKEN`.

In [1]:
!pip install trl

In [2]:
import os
import torch
import json
import shutil
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

import huggingface_hub
from kaggle_secrets import UserSecretsClient

# Authenticate using Colab Secrets
user_secrets = UserSecretsClient()
huggingface_hub.login(user_secrets.get_secret("HF_TOKEN"))

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Section 2 — Configuration
Define dataset path and output model path. `google/gemma-2b-it` easily fits within 4GB VRAM using 4-bit quantization. If using Kaggle T4 x2, `device_map='auto'` automatically utilizes available GPUs.

In [3]:
# --- CONFIGURATION FOR KAGGLE ---

DATASET_PATH = "/kaggle/input/datasets/chattimohamedlouay/dlp-dataset/dlp_ml_dataset.jsonl"
OUTPUT_DIR = "/kaggle/working/dlp_lora_outputs"
MODEL_PATH = "/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2"


# Ensure saving directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Section 3 — Load Dataset
Load the `.jsonl` file and convert it to a HuggingFace Dataset format (all 6000 samples for training).

In [4]:
def load_jsonl(file_path):
    data = []
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                data.append(json.loads(line))
        print(f"Loaded {len(data)} samples from {file_path}")

dataset_list = load_jsonl(DATASET_PATH)
dataset = Dataset.from_list(dataset_list)

print("\nDataset length:", len(dataset))
print("Sample keys:", dataset[0].keys())

Loaded 6000 samples from /kaggle/input/datasets/chattimohamedlouay/dlp-dataset/dlp_ml_dataset.jsonl

Dataset length: 6000
Sample keys: dict_keys(['text', 'surface', 'label', 'features'])


## Section 4 — Preprocessing
Convert each sample into the structured instruction prompt. The model will learn to output strictly the target label: `ALLOW`, `REDACT`, `ESCALATE`, or `BLOCK`.

In [5]:
def format_dlp_prompt(example):
    # Extract features safely
    features = example.get("features", {})
    features_str = "\n".join([f"{k}={v}" for k, v in features.items()])

    # Construct the strictly formatted prompt matching the new detailed requirement
    prompt = f"""You are a Data Leakage Prevention (DLP) classifier for AI agents.

Your task is to analyze the input and classify its risk level into EXACTLY one of:
ALLOW, REDACT, ESCALATE, BLOCK.

You are given:
- SURFACE: where the data appears (OUTPUT: scanning the agent's final output to the user, TOOL_ARGS: scanning the agent's arguments in a tool call, TOOL_RESULT: scanning the results of a tool call)
- FEATURES: extracted signals from deterministic analysis
- TEXT: the raw content

You must follow these rules:

DEFINITIONS:

- ALLOW:
  Content is safe. It may include examples, placeholders, documentation, or synthetic data.
  No real sensitive data is exposed.

- REDACT:
  Contains limited personal data (e.g., one or a few emails, phone numbers, or credit cards).
  Data is real or realistic but not highly sensitive.

- ESCALATE:
  Content is ambiguous or potentially risky.
  Examples:
    - multiple PII entries
    - high-entropy strings that may be tokens
    - partial or unclear secrets
    - mixed benign and sensitive signals
  If you are unsure, choose ESCALATE.

- BLOCK:
  Content clearly contains highly sensitive data or secrets.
  Examples:
    - API keys, tokens, credentials
    - database connection strings with passwords
    - private keys
    - authentication headers
  Always BLOCK if a real secret is present.

FEATURE USAGE GUIDELINES:

- num_emails > 0 → indicates presence of PII (REDACT or ESCALATE depending on amount)
- num_phones > 0 → PII signal (REDACT or ESCALATE)
- num_credit_cards > 0 → sensitive PII (REDACT or ESCALATE)
- has_valid_credit_card = true → strong sensitive signal (prefer REDACT or ESCALATE)
- num_secrets_detected > 0 → strong signal for BLOCK
- has_api_key_pattern = true → strong signal for BLOCK
- has_db_connection = true → strong signal for BLOCK (especially if credentials present)
- has_private_key = true → definite BLOCK
- high max_entropy or avg_entropy → may indicate tokens/secrets → ESCALATE or BLOCK depending on context
- is_example_context = true → reduces risk (may allow ALLOW)
- is_code_context = true → may indicate examples or documentation (reduce risk unless real secrets present)
- multiple risk signals combined → increase severity

PRIORITY RULES:

1. If real secrets or credentials are present → BLOCK
2. Else if clear PII is present → REDACT
3. Else if ambiguous or suspicious → ESCALATE
4. Else → ALLOW

IMPORTANT:

- Use TEXT as the primary source of truth
- Use FEATURES as supporting signals, not absolute truth
- If conflicting signals exist, prioritize safety
- If unsure, choose ESCALATE
- Do NOT explain your answer
- Output ONLY one word: ALLOW, REDACT, ESCALATE, or BLOCK

### Input:
[SURFACE={example.get('surface', 'UNKNOWN')}]

[FEATURES]
{features_str}

[TEXT]
{example.get('text', '')}

### Output:
{example.get('label', 'UNKNOWN')}"""

    return {"formatted_prompt": prompt}

# Apply formatting mapping across whole dataset
dataset = dataset.map(format_dlp_prompt)

print("\n--- Example Formatted Target Prompt ---\n")
print(dataset[0]["formatted_prompt"])

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]


--- Example Formatted Target Prompt ---

You are a Data Leakage Prevention (DLP) classifier for AI agents.

Your task is to analyze the input and classify its risk level into EXACTLY one of:
ALLOW, REDACT, ESCALATE, BLOCK.

You are given:
- SURFACE: where the data appears (OUTPUT: scanning the agent's final output to the user, TOOL_ARGS: scanning the agent's arguments in a tool call, TOOL_RESULT: scanning the results of a tool call)
- FEATURES: extracted signals from deterministic analysis
- TEXT: the raw content

You must follow these rules:

DEFINITIONS:

- ALLOW:
  Content is safe. It may include examples, placeholders, documentation, or synthetic data.
  No real sensitive data is exposed.

- REDACT:
  Contains limited personal data (e.g., one or a few emails, phone numbers, or credit cards).
  Data is real or realistic but not highly sensitive.

- ESCALATE:
  Content is ambiguous or potentially risky.
  Examples:
    - multiple PII entries
    - high-entropy strings that may be to

## Section 5 — Tokenization
Load the tokenizer and ensure padding and truncation are handled so sequence lengths are uniform. We use SFTTrainer which handles the tokenization step internally via `dataset_text_field`.

In [6]:
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Gemma and Llama models often require this setup for padding to prevent sequence length issues
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

## Section 6 — Model Loading
Load the base model using 4-bit float quantization (`BitsAndBytesConfig`). Also enable gradient checkpointing to drastically save memory during backpropagation.

In [7]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [8]:
!pip install -U bitsandbytes>=0.46.1

In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16  # ← change float16 to bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map={"":0},
    dtype=torch.bfloat16,  # ← change to bfloat16
)
model.config.use_cache = False # Required for gradient checkpointing

# Prepare for custom k-bit training, drastically saving VRAM
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

## Section 7 — Apply LoRA
Attach Low-Rank Adaptation (LoRA) adapters to the attention blocks. This freezes the base model and only updates new compact weights.

In [10]:
# Configure LoRA Parameters
lora_config = LoraConfig(
    r=8,                  # Rank
    lora_alpha=32,        # Scale
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # Target attention heads for Gemma/Llama
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach adapters to the model
model = get_peft_model(model, lora_config)

# Verify we only train ~0.1% parameters
model.print_trainable_parameters()

trainable params: 3,194,880 || all params: 2,617,536,768 || trainable%: 0.1221


## Section 8 & 9 — Training Setup & Execution
Define hyperparameters suitable for a T4 setup. Small batch size + gradient accumulation steps = stable simulated large batch. Train for 1 epoch and max 300 steps as requested.

In [11]:
from trl import SFTConfig, SFTTrainer

# Configure Training Hyperparameters specifically built for ~4GB VRAM
training_args = SFTConfig(
    output_dir="./lora_checkpoints",
    # CRITICAL: Batch size MUST be 1 for 4GB VRAM! Using 2 will instantly cause CUDA OOM.
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # 1 x 8 = Effective batch size of 8 still works fine
    max_steps=300,
    learning_rate=2e-4,
    fp16=False,   # ← disable fp16
    bf16=True,    # ← enable bf16
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    optim="paged_adamw_8bit",        # This explicitly tells PyTorch to offload memory spikes to CPU RAM!
    dataset_text_field="formatted_prompt",
    max_length=1024,                  # Reduce to 256 or 384 ONLY if you still encounter OOM
    gradient_checkpointing=True,     # Unloads forward activations to save major VRAM
    gradient_checkpointing_kwargs={'use_reentrant': False} # Safe for PEFT overlapping
)

# Initialize Supervised Fine-Tuning Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

print("\nStarting Local LoRA Fine-Tuning...\n")
# Execute Training Locally
trainer.train()

Adding EOS to train dataset:   0%|          | 0/6000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.



Starting Local LoRA Fine-Tuning...



Step,Training Loss
10,1.326105
20,0.335322
30,0.209061
40,0.186511
50,0.193811
60,0.179621
70,0.177241
80,0.172687
90,0.162884
100,0.148520


TrainOutput(global_step=300, training_loss=0.19944422443707785, metrics={'train_runtime': 37226.4225, 'train_samples_per_second': 0.129, 'train_steps_per_second': 0.008, 'total_flos': 5.373664316859802e+16, 'train_loss': 0.19944422443707785})

## Section 10 — Save Model
Save the fine-tuned LoRA adapters + Tokenizer. The output is tiny (approx 50-100MB).

In [12]:
print(f"Applying final save to {OUTPUT_DIR}...")

# Save PEFT model adapters
trainer.model.save_pretrained(OUTPUT_DIR)

# Save the tokenizer
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Model adapters and tokenizer successfully saved.")

Applying final save to /kaggle/working/dlp_lora_outputs...
✅ Model adapters and tokenizer successfully saved.


## Section 11 — Export / Download
Zip the output folder containing the adapters, and generate a Kaggle direct download link.

In [13]:
# Create a Zip file of the exported directory
zip_path = "dlp_lora_package"
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f"Archived model to {zip_path}.zip")

# Note: The zip file is generated neatly in your local workspace folder.

Archived model to dlp_lora_package.zip
